# 00 Infrastructure and port check

This notebook checks that PAGO can run the Haenam detection pipeline for the 2025 Imsil swarm before any data are processed.
It answers four questions:

1. **Environments.** Do the three rebuilt Python environments (`dl-base`, `pq-gpu`, `fmf`) carry the same package versions as moho?
2. **Smoke tests.** Do the picker, the associator, HYPOINVERSE, hypoDD, the relocation framework and the template matcher run on PAGO and reproduce known results?
3. **Port check.** Is the ported code identical to the Haenam code, apart from the region values, paths and labels listed in the decision log?
4. **Provenance.** Where did each copied file come from (commit, checksum)?

Nothing here changes any file; the cells only read logs and run read-only checks.

In [1]:
import os, re, glob, subprocess
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 120)
PROJECT = '/home/msseo/works/72.2025_Imsil_swarm'
DL = f'{PROJECT}/DL_detection'
EXT = f'{PROJECT}/_external'
MF = '/home/msseo/miniforge3'
checks = []   # (check, status, detail) rows for the summary


def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

## Environments

`envs/verify_envs.py` compares each rebuilt environment with moho's exported package list (2026-10-04).
The key packages, the ones the pipeline imports, must match exactly. All other packages are only counted.

In [2]:
rc, txt = run(f'{MF}/bin/python {PROJECT}/envs/verify_envs.py')
print(txt)
checks.append(('Environments match moho (key packages)', 'pass' if rc == 0 else 'FAIL', txt.strip().splitlines()[-1] if txt.strip() else 'no output'))

[dl-base] torch CUDA 13.0, GPU NVIDIA A100-PCIE-40GB
    ok  torch                moho 2.12.0   PAGO 2.12.0
    ok  torchvision          moho 0.27.0   PAGO 0.27.0
    ok  seisbench            moho 0.11.0   PAGO 0.11.0
    ok  pyocto               moho 0.1.10   PAGO 0.1.10
    ok  fsspec               moho 2026.1.0   PAGO 2026.1.0
    ok  numba                moho 0.63.1   PAGO 0.63.1
    ok  numpy                moho 2.3.5   PAGO 2.3.5
    ok  pandas               moho 2.3.3   PAGO 2.3.3
    ok  h5py                 moho 3.15.1   PAGO 3.15.1
    ok  obspy                moho 1.4.2   PAGO 1.4.2
    ok  scipy                moho 1.17.0   PAGO 1.17.0
    ok  pyrocko              moho 2025.12.04   PAGO 2025.12.4
    ok  matplotlib           moho 3.10.8   PAGO 3.10.8
    ok  pygmt                moho 0.18.0   PAGO 0.18.0
    other exported packages: 53 same, 48 different version, 380 not installed (informative)
[pq-gpu] torch CUDA 12.8, GPU NVIDIA A100-PCIE-40GB
    ok  torch               

## Smoke tests

`tools/smoke_envs.sh` runs eight short tests in a scratch directory. The two most informative ones reproduce earlier results byte for byte:
the ML 1.9 HYPOINVERSE solution of `1.HypoInv/kim2011` (35.5506° N, 127.2951° E, 14.40 km). The hypoDD example must agree with moho's run within 1 m and 0.01 s: the binary is the same, but the system math library differs between the two machines, so the last digits differ.
The cell shows the latest log.

In [3]:
logs = sorted(glob.glob(f'{DL}/logs/smoke_envs_*.txt'))
if logs:
    txt = open(logs[-1]).read()
    print(os.path.basename(logs[-1])); print('\n'.join(l for l in txt.splitlines() if l.startswith(('===', '    PASS', '    FAIL', 'RESULT'))))
    m = re.search(r'RESULT: (\d+) passed, (\d+) failed', txt)
    status = 'pass' if m and m.group(2) == '0' else 'FAIL'
    checks.append(('Smoke tests (tools/smoke_envs.sh)', status, m.group(0) if m else 'no RESULT line'))
else:
    print('no smoke log yet: run tools/smoke_envs.sh'); checks.append(('Smoke tests (tools/smoke_envs.sh)', 'not run', '-'))

smoke_envs_20261007_1851.txt
=== S1 GPU visible + matmul (dl-base, pq-gpu)
    PASS
=== S2 PhaseNet+ weights sha256 + model load (dl-base)
    PASS
=== S3 PyOcto velocity model + REGION_STRICT associator (dl-base)
    PASS
=== S4 hyp1.40 wrapper reproduces 1.HypoInv/kim2011/Imsil.sum byte-for-byte
    PASS
=== S5 hypoDD v2.1beta example1 on PAGO = moho within 1 m / 0.01 s
    PASS
=== S6 korea-cluster-relocation imports + Imsil cluster config (pq-gpu)
    PASS
=== S7 fast_matched_filter GPU/CPU loaded, synthetic self-detection, GPU = CPU (fmf)
    PASS
=== S8 Helvetica + PyGMT render (dl-base)
    PASS
RESULT: 8 passed, 0 failed   (log /home/msseo/works/72.2025_Imsil_swarm/DL_detection/logs/smoke_envs_20261007_1851.txt; scratch /tmp/imsil_smoke.xVCj)


## Port check against the Haenam code

`tools/port_check.sh` compares `DL_detection` with the read-only Haenam copy:

- **Must be identical:** the picker and association code (`core.py`, `pick_augmentation.py`, ...), the velocity models, the SAC export, the magnitude engine and the FMF scan code.
- **Whitelisted:** region values, paths and labels. Their full diffs are written to `logs/port_diff_vs_haenam.txt`.
- **Other files:** listed; a modified copy that is not whitelisted fails the check.

The `pipeline/config.py` diff below shows every region value that changed.

In [4]:
rc, txt = run(f'{PROJECT}/tools/port_check.sh')
print(txt)
checks.append(('Port check (identical core, whitelisted diffs only)', 'pass' if rc == 0 else 'FAIL', [l for l in txt.splitlines() if l.startswith('RESULT')][0] if 'RESULT' in txt else 'no RESULT'))
rep = open(f'{DL}/logs/port_diff_vs_haenam.txt').read()
sec = rep.split('---------------------------------------------------------------- pipeline/config.py')[1].split('----------------------------------------------------------------')[0]
print('pipeline/config.py vs Haenam (changed lines only):')
print('\n'.join(l for l in sec.splitlines() if l.startswith(('+', '-')) and not l.startswith(('+++', '---'))))

== 1. must-be-identical (sha256)
== 2. whitelist diffs (region values / paths only)
---------------------------------------------------------------- pipeline/config.py   (vs Haenam pipeline/config.py)
---------------------------------------------------------------- run_imsil.sh   (vs Haenam run_haenam.sh)
---------------------------------------------------------------- tools/mirror_mara.py   (vs Haenam tools/mirror_mara.py)
---------------------------------------------------------------- tools/link_continuous.py   (vs Haenam tools/link_continuous.py)
---------------------------------------------------------------- tools/run_status.py   (vs Haenam tools/run_status.py)
---------------------------------------------------------------- station_table/build_stations.py   (vs Haenam station_table/build_stations.py)
---------------------------------------------------------------- station_table/build_station_epochs.py   (vs Haenam station_table/build_station_epochs.py)
--------------------------

## Copied code, weights and binaries

`tools/phase0_copy_from_moho.sh` records every copy in `_external/PROVENANCE.tsv` and saves moho's `git diff` of each code tree in `_external/provenance/`.
The PhaseNet+ weights checksum must equal the one frozen in Haenam's `logs/full_run_params.json`.

In [5]:
prov_path = f'{EXT}/PROVENANCE.tsv'
if os.path.exists(prov_path):
    prov = pd.read_csv(prov_path, sep='\t'); display(prov)
    w = prov[prov['item'] == 'model_99.pth']
    ok = len(w) and w.checksum.iloc[-1].endswith('f1643330e34bffaa4f7e3a5f7dd9205fb4019d76e5b9b6cb598232360d22f384')
    checks.append(('PhaseNet+ weights = Haenam frozen weights', 'pass' if ok else 'FAIL', w.checksum.iloc[-1][:30] + '...' if len(w) else 'missing'))
else:
    print('no provenance yet: run tools/phase0_copy_from_moho.sh run'); checks.append(('PhaseNet+ weights = Haenam frozen weights', 'not run', '-'))
for b in ['hypoDD', 'hyp1.40', 'ph2dt', 'ncsn2pha']:
    p = f'{EXT}/bin/{b}'
    print(f'{b:9s}', run(f'md5sum {p} 2>/dev/null; head -c 300 {p} | grep -a exec')[1].strip().replace(chr(10), ' | ') if os.path.exists(p) else 'missing')

,item,source,commit,checksum,bytes,date
0,EQNet,moho:/home/msseo/works/14.EQNet/EQNet,ad492ef0fe93493662d324f7747851be2c9fc7b8,tree (diff: provenance/EQNet.diff),85381427,2026-10-07 18:11
1,korea-cluster-relocation,moho:/home/msseo/works/15.PocketQuake/external/korea-cluster-relocation,663c26bcdcbcd0440e7682ec7298a8046764bc49,tree (diff: provenance/korea-cluster-relocation.diff),1231848,2026-10-07 18:11
2,fast_matched_filter,moho:/home/msseo/works/FMF/fast_matched_filter,25e8263eadf9d9c32b6da15b93c7a5cb930cd56f,tree (diff: provenance/fast_matched_filter.diff),21368200,2026-10-07 18:11
3,HypoDD,moho:/home/msseo/Downloads/HypoDD,fb40407fb768cbdc23a11e27bc841f5d4a98abb7,tree (diff: provenance/HypoDD.diff),61831425,2026-10-07 18:11
4,model_99.pth,moho:/home/msseo/works/14.EQNet/EQNet/docs/model_phasenet_plus/model_99.pth,-,sha256:f1643330e34bffaa4f7e3a5f7dd9205fb4019d76e5b9b6cb598232360d22f384,18810308,2026-10-07 18:11
5,hypoDD,moho:/home/msseo/bin/hypoDD,fb40407+diff,md5:e36d745c61a6f4508fd6c0dbab89e7c7,227696,2026-10-07 18:11
6,hyp1.40 (wrapper),PAGO:/home/msseo/bin/hyp1.40,-,md5:dd771a6d51a0f61e714f6fc79ad99d35,2387432,2026-10-07 18:11
7,ph2dt (wrapper),PAGO:/home/msseo/bin/ph2dt,-,md5:91e9582c38146121330d53610812d2d0,44696,2026-10-07 18:11
8,ncsn2pha (wrapper),PAGO:/home/msseo/bin/ncsn2pha,-,md5:614663792bf3a21a2bec048dfcc27464,22352,2026-10-07 18:11
9,kma_entire_catalog,moho:/home/msseo/works/07.SeismoStats/catalog/kma_entire_catalog_260309.csv,-,sha256:56b7600cf0ea821fdd1f33a4be10e0ca2e8b86fe814a250010fcffae1f3ab5e5,662142,2026-10-07 18:11


hypoDD    e36d745c61a6f4508fd6c0dbab89e7c7  /home/msseo/works/72.2025_Imsil_swarm/_external/bin/hypoDD
hyp1.40   1989f4c6fcfa712bd0d9aa4f840ecc2e  /home/msseo/works/72.2025_Imsil_swarm/_external/bin/hyp1.40 | LD_LIBRARY_PATH=/home/msseo/works/72.2025_Imsil_swarm/_external/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH} exec /home/msseo/bin/hyp1.40 "$@"
ph2dt     634ecd3c1a92d26ca0a3304c76fcfe5e  /home/msseo/works/72.2025_Imsil_swarm/_external/bin/ph2dt | LD_LIBRARY_PATH=/home/msseo/works/72.2025_Imsil_swarm/_external/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH} exec /home/msseo/bin/ph2dt "$@"
ncsn2pha  b4a64150ba11d1fd1352114f49728284  /home/msseo/works/72.2025_Imsil_swarm/_external/bin/ncsn2pha | LD_LIBRARY_PATH=/home/msseo/works/72.2025_Imsil_swarm/_external/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH} exec /home/msseo/bin/ncsn2pha "$@"


## Decision log

Every setting that differs from Haenam, and every open choice, is listed in `docs/decisions.csv` with its default.
The **rule** rows change a value by a stated rule, for example the same box half-widths centred on Imsil.
The **open** rows wait for evidence from the stage notebooks and for your decision.

In [6]:
dec = pd.read_csv(f'{DL}/docs/decisions.csv')
display(dec[['id', 'setting', 'haenam_value', 'imsil_default', 'status', 'evidence']])
n_open = int((dec.status == 'open').sum())
checks.append(('Decision log', 'info', f"{len(dec)} rows: {int((dec.status == 'rule').sum())} rule, {int((dec.status == 'decided').sum())} decided, {int((dec.status == 'stated').sum())} stated, {n_open} open"))

,id,setting,haenam_value,imsil_default,status,evidence
0,P1,detection period,2020-01-01 .. 2026-09-22 (full period),2025 DOY 244-365 + 2026 DOY 1-31 (Sep 2025 - Jan 2026),decided,user 2026-10-07
1,P2,run machine,moho,PAGO (envs rebuilt from moho specs),decided,user 2026-10-07
2,P3,MARA transfer,moho pulls from MARA,PAGO pulls directly (user adds PAGO key to MARA),decided,user 2026-10-07
3,R1,PyOcto association box,"lat 33.95-35.35, lon 125.60-127.20 (+/-0.70 / +/-0.80 deg around Haenam)","lat 34.85-36.25, lon 126.50-128.10 (same half-widths around Imsil)",rule,-
4,R2,source epicentre,"34.660, 126.405","35.5506, 127.2951 (ML 1.9 hand-pick HYPOINVERSE, kim2011)",rule,1.HypoInv/kim2011/Imsil.sum
5,R3,relocation box / in-box flag,"34.55-34.77, 126.28-126.53 (+/-0.11 / +/-0.125 deg)","35.44-35.66, 127.17-127.42 (same half-widths)",rule,-
6,R4,station table,KS/KG within 100 km of Haenam (+ JU),KS within 100 km of Imsil (station_table/used_stations_100km.csv; 55),rule,-
7,R5,known-event table (template_match),195-event relocated Haenam catalog,24 KMA-listed Imsil events (UTC; KMA ML),rule,-
8,R6,ML response epoch window,2020-01-01 .. 2026-09-12,2025-09-01 .. 2026-02-01,rule,-
9,D1,station elevations in stations.csv,0.0 m for all KS/KG rows,0.0 m (Haenam and existing Imsil STA convention); alternative: real elevations 42-698 m,open,nb02: smoke days located both ways (event count; epicentre/depth shifts)


## Comprehensive summary

In [7]:
summary = pd.DataFrame(checks, columns=['check', 'status', 'detail'])
display(summary)
ready = all(s in ('pass', 'info') for s in summary.status)
print('Ready for stage S1 (stations and data mirror):', 'yes' if ready else 'no, fix the rows above that do not pass')
print()
st = dict(zip(summary.check, summary.status))
smoke_txt = open(logs[-1]).read() if logs else ''
print('Take-home points (from the checks above)')
print('  - Port check:', 'the Imsil code equals the Haenam code apart from the whitelisted region values, paths and labels in docs/decisions.csv.'
      if st.get('Port check (identical core, whitelisted diffs only)') == 'pass' else 'NOT clean; see logs/port_diff_vs_haenam.txt.')
print('  - Environments:', 'key package versions equal moho.' if st.get('Environments match moho (key packages)') == 'pass' else 'not yet equal to moho (see the environment table).')
s4 = re.search(r'=== S4.*?\n    (PASS|FAIL)', smoke_txt, re.S); s5 = re.search(r'=== S5.*?\n    (PASS|FAIL)', smoke_txt, re.S)
print('  - ML 1.9 HYPOINVERSE solution reproduced on PAGO (S4):', s4.group(1) if s4 else 'not run',
      '| hypoDD example = moho within 1 m (S5):', s5.group(1) if s5 else 'not run')
print(f'  - Decision log: {n_open} open decisions; they are settled in the stage notebooks 01-06.')
print()
print('Reproduce:  tools/phase0_copy_from_moho.sh run; envs/build_envs.sh all run; tools/smoke_envs.sh; then re-run this notebook.')
print('Next:       ./run_imsil.sh stations run, then the mirror (notebook 01).')

,check,status,detail
0,Environments match moho (key packages),pass,RESULT: PASS
1,Smoke tests (tools/smoke_envs.sh),pass,"RESULT: 8 passed, 0 failed"
2,"Port check (identical core, whitelisted diffs only)",pass,RESULT: PASS
3,PhaseNet+ weights = Haenam frozen weights,pass,sha256:f1643330e34bffaa4f7e3a5...
4,Decision log,info,"25 rows: 6 rule, 3 decided, 4 stated, 12 open"


Ready for stage S1 (stations and data mirror): yes

Take-home points (from the checks above)
  - Port check: the Imsil code equals the Haenam code apart from the whitelisted region values, paths and labels in docs/decisions.csv.
  - Environments: key package versions equal moho.
  - ML 1.9 HYPOINVERSE solution reproduced on PAGO (S4): PASS | hypoDD example = moho within 1 m (S5): PASS
  - Decision log: 12 open decisions; they are settled in the stage notebooks 01-06.

Reproduce:  tools/phase0_copy_from_moho.sh run; envs/build_envs.sh all run; tools/smoke_envs.sh; then re-run this notebook.
Next:       ./run_imsil.sh stations run, then the mirror (notebook 01).
